# Setup

In [ ]:
%matplotlib inline

import os, sys, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import uniform_filter1d

import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.rcParams.update({"font.size": 8})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='Arial')
import seaborn as sns
sns.set_style("white")

import src.utils as utils

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

def set_font_size(font_size=12):
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': font_size+1,
        'axes.labelsize': font_size,
        'xtick.labelsize': font_size-1,
        'ytick.labelsize': font_size-1,
        'legend.fontsize': font_size,
        'figure.titlesize': font_size+2,
    })

# Load data

In [ ]:
# path to the pickle file produced by biornn_results_dynamics_trajectory.py
model_params_name = 'model_params_202603bp'
username = os.getenv('USER')
if sys.platform == 'darwin':
    if username == 'ahmad':
        datadir = '/Users/ahmad/software/snaplab_github/neuro_rnn/data'
        modeldir = os.path.join('/Volumes/Sabrent_2TB/rutgers/neuro_rnn/data', model_params_name)
elif sys.platform == 'linux':
    if username == 'ab2792':
        datadir = '/home/ab2792/software/snaplab_github/neuro_rnn/data'
        modeldir = '/home/ab2792/data/neuro_rnn/results/pytorch/model'
    elif username == 'lindenmp':
        datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
        modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/model_cpu'

results_file = os.path.join(modeldir, f'training_trajectory_{model_params_name}.pkl')
with open(results_file, 'rb') as f:
    data = pickle.load(f)

save_figs = False
outdir = modeldir

print(f'Loaded: {results_file}')
print(f'Models: {len(data["results"])}')
print(f'Args: {data["args"]}')

# Unpack results

In [ ]:
model_params = data['model_params']
fmri_baselines = data['fmri_intersubj_baselines']
results = data['results']
n_models = len(results)

# derive task/kernel structure
task_names = list(model_params.task_no_modifier.unique())
n_tasks = len(task_names)
kernel_labels = list(model_params.kernel_label.unique())
n_kernels = len(kernel_labels)

# color palette
if n_kernels <= 3:
    color_palette = utils.get_my_colors(cat_trio=True, as_list=True)
else:
    color_palette = utils.get_my_colors(cat_trio=False, as_list=True)
color_palette = color_palette[:n_kernels]

print(f'Tasks: {task_names}')
print(f'Kernels: {kernel_labels}')
if fmri_baselines['task'] is not None:
    print(f'fMRI baselines — task: {fmri_baselines["task"]:.4f}, rest: {fmri_baselines["rest"]:.4f}')

# Accuracy over training

In [ ]:
size_scale = 2.0
font_size = int(6 * size_scale)
set_font_size(font_size)
n_fig_columns = min(3, n_tasks)
n_fig_rows = int(np.ceil(n_tasks / n_fig_columns))
fig_size_w = 3 * n_fig_columns * size_scale
fig_size_h = 2 * n_fig_rows * size_scale

fig, _ = plt.subplots(n_fig_rows, n_fig_columns, figsize=(fig_size_w, fig_size_h),
                       squeeze=True, sharex=True, sharey=True)

for model_idx in range(n_models):
    res = results[model_idx]
    if res is None:
        continue

    this = model_params.iloc[model_idx]
    task_index = this.task_index
    kernel_index = this.kernel_index
    ax = fig.axes[task_index]

    epochs = np.array(res['sampled_epochs'])
    n_runs = res['n_runs']
    acc = np.zeros((n_runs, len(epochs)))

    for ri, run in enumerate(res['runs']):
        for ei, ep in enumerate(epochs):
            acc[ri, ei] = run['epochs'][ep]['accuracy']

    acc_mean = acc.mean(axis=0)
    acc_se = acc.std(axis=0) / np.sqrt(n_runs)

    ax.plot(epochs, acc_mean, color=color_palette[kernel_index],
            label=this.kernel_label, linewidth=2)
    ax.fill_between(epochs, acc_mean - 1.96 * acc_se, acc_mean + 1.96 * acc_se,
                     color=color_palette[kernel_index], alpha=0.1)
    ax.set_title(this.task_label)

fig.axes[0].set_ylim((-0.05, 1.05))
# legend on first subplot only, deduplicated
handles, labels = fig.axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.axes[0].legend(by_label.values(), by_label.keys(), loc='lower right')

fig.text(0.51, 0.0, 'Epoch', ha='center', va='center')
fig.text(0.0, 0.5, 'Test Accuracy', ha='center', va='center', rotation='vertical')
sns.despine(fig=fig, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
fig.tight_layout()

if save_figs:
    fig.savefig(os.path.join(outdir, 'trajectory_accuracy.svg'),
                dpi=300, bbox_inches='tight', pad_inches=0.01)
plt.show()

# Weight-kernel similarity over training

In [ ]:
similarity_metrics = {
    'spearman': 'Weight–kernel similarity (Spearman r)',
    'cosine': 'Weight–kernel similarity (Cosine)',
}

for metric_key, metric_label in similarity_metrics.items():

    size_scale = 2.0
    font_size = int(6 * size_scale)
    set_font_size(font_size)
    fig, _ = plt.subplots(n_fig_rows, n_fig_columns, figsize=(fig_size_w, fig_size_h),
                           squeeze=True, sharex=True, sharey=True)

    for model_idx in range(n_models):
        res = results[model_idx]
        if res is None:
            continue

        this = model_params.iloc[model_idx]
        task_index = this.task_index
        kernel_index = this.kernel_index
        ax = fig.axes[task_index]

        # skip models without spatial kernel
        if res['runs'][0]['epochs'][res['sampled_epochs'][0]]['weight_kernel_similarity'] is None:
            continue

        epochs = np.array(res['sampled_epochs'])
        n_runs = res['n_runs']
        vals = np.zeros((n_runs, len(epochs)))

        for ri, run in enumerate(res['runs']):
            for ei, ep in enumerate(epochs):
                vals[ri, ei] = run['epochs'][ep]['weight_kernel_similarity'][metric_key]

        vals_mean = vals.mean(axis=0)
        vals_se = vals.std(axis=0) / np.sqrt(n_runs)

        ax.plot(epochs, vals_mean, color=color_palette[kernel_index],
                label=this.kernel_label, linewidth=2)
        ax.fill_between(epochs, vals_mean - 1.96 * vals_se, vals_mean + 1.96 * vals_se,
                         color=color_palette[kernel_index], alpha=0.1)
        ax.set_title(this.task_label)

    handles, labels = fig.axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.axes[0].legend(by_label.values(), by_label.keys(), loc='lower right')

    fig.text(0.51, 0.0, 'Epoch', ha='center', va='center')
    fig.text(0.0, 0.5, metric_label, ha='center', va='center', rotation='vertical')
    sns.despine(fig=fig, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
    fig.tight_layout()

    if save_figs:
        fig.savefig(os.path.join(outdir, f'trajectory_wk_{metric_key}.svg'),
                    dpi=300, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# fMRI variance explained over training

In [ ]:
fmri_ve_metrics = [
    {
        'key': 'fmri_task_ve',
        'baseline': fmri_baselines['task'],
        'ylabel': 'Task fMRI variance explained\n(fraction of inter-subject baseline)',
        'file_suff': 'fmri_task_ve',
    },
    {
        'key': 'fmri_rest_ve',
        'baseline': fmri_baselines['rest'],
        'ylabel': 'Rest fMRI variance explained\n(fraction of inter-subject baseline)',
        'file_suff': 'fmri_rest_ve',
    },
]

for vm in fmri_ve_metrics:
    if vm['baseline'] is None:
        print(f'Skipping {vm["key"]} (no fMRI data)')
        continue

    size_scale = 2.0
    font_size = int(6 * size_scale)
    set_font_size(font_size)
    fig, _ = plt.subplots(n_fig_rows, n_fig_columns, figsize=(fig_size_w, fig_size_h),
                           squeeze=True, sharex=True, sharey=True)

    for model_idx in range(n_models):
        res = results[model_idx]
        if res is None:
            continue

        this = model_params.iloc[model_idx]
        task_index = this.task_index
        kernel_index = this.kernel_index
        ax = fig.axes[task_index]

        epochs = np.array(res['sampled_epochs'])
        n_runs = res['n_runs']
        vals = np.zeros((n_runs, len(epochs)))

        for ri, run in enumerate(res['runs']):
            for ei, ep in enumerate(epochs):
                ve_arr = run['epochs'][ep][vm['key']]
                if ve_arr is None:
                    vals[ri, ei] = np.nan
                else:
                    # sum across PCs, mean across subjects, normalize by baseline
                    vals[ri, ei] = ve_arr.sum(axis=0).mean() / vm['baseline']

        vals_mean = np.nanmean(vals, axis=0)
        vals_se = np.nanstd(vals, axis=0) / np.sqrt(n_runs)

        ax.plot(epochs, vals_mean, color=color_palette[kernel_index],
                label=this.kernel_label, linewidth=2)
        ax.fill_between(epochs, vals_mean - 1.96 * vals_se, vals_mean + 1.96 * vals_se,
                         color=color_palette[kernel_index], alpha=0.1)
        ax.set_title(this.task_label)

    handles, labels = fig.axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.axes[0].legend(by_label.values(), by_label.keys(), loc='lower right')

    fig.text(0.51, 0.0, 'Epoch', ha='center', va='center')
    fig.text(0.0, 0.5, vm['ylabel'], ha='center', va='center', rotation='vertical')
    sns.despine(fig=fig, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
    fig.tight_layout()

    if save_figs:
        fig.savefig(os.path.join(outdir, f'trajectory_{vm["file_suff"]}.svg'),
                    dpi=300, bbox_inches='tight', pad_inches=0.01)
    plt.show()

# Joint trajectory: weight-kernel similarity vs fMRI VE

In [ ]:
# Scatter: weight-kernel similarity (x) vs fMRI VE (y), color-coded by epoch
# One subplot per (task × kernel), only for models with spatial kernels

if fmri_baselines['task'] is None:
    print('Skipping (no fMRI data)')
else:
    wk_metric = 'spearman'
    fmri_key = 'fmri_task_ve'
    fmri_baseline = fmri_baselines['task']

    size_scale = 2.0
    font_size = int(6 * size_scale)
    set_font_size(font_size)

    for task_idx in range(n_tasks):
        # collect models for this task that have spatial kernels
        task_models = []
        for model_idx in range(n_models):
            res = results[model_idx]
            if res is None:
                continue
            this = model_params.iloc[model_idx]
            if this.task_index != task_idx:
                continue
            ep0 = res['sampled_epochs'][0]
            if res['runs'][0]['epochs'][ep0]['weight_kernel_similarity'] is None:
                continue
            task_models.append((model_idx, this))

        if not task_models:
            continue

        n_panels = len(task_models)
        fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels * size_scale, 3 * size_scale),
                                  squeeze=False, sharey=True)
        axes = axes.ravel()

        for pi, (model_idx, this) in enumerate(task_models):
            res = results[model_idx]
            ax = axes[pi]
            epochs = np.array(res['sampled_epochs'])
            n_runs = res['n_runs']

            # colormap: epoch progression
            cmap = cm.viridis
            norm = plt.Normalize(epochs.min(), epochs.max())

            for ri, run in enumerate(res['runs']):
                wk_vals = []
                ve_vals = []
                for ep in epochs:
                    ep_data = run['epochs'][ep]
                    wk_vals.append(ep_data['weight_kernel_similarity'][wk_metric])
                    ve_arr = ep_data[fmri_key]
                    if ve_arr is not None:
                        ve_vals.append(ve_arr.sum(axis=0).mean() / fmri_baseline)
                    else:
                        ve_vals.append(np.nan)

                wk_vals = np.array(wk_vals)
                ve_vals = np.array(ve_vals)

                # plot trajectory as connected scatter
                for ei in range(len(epochs) - 1):
                    ax.plot(wk_vals[ei:ei+2], ve_vals[ei:ei+2],
                            color=cmap(norm(epochs[ei])), linewidth=0.5, alpha=0.3)
                ax.scatter(wk_vals, ve_vals, c=epochs, cmap=cmap, norm=norm,
                          s=15, alpha=0.5, edgecolors='none', zorder=3)

            ax.set_xlabel(f'Weight–kernel similarity ({wk_metric})')
            if pi == 0:
                ax.set_ylabel('Task fMRI VE\n(frac. of baseline)')
            ax.set_title(this.kernel_label)

        # colorbar on last axis
        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axes[-1], fraction=0.05, pad=0.02)
        cbar.set_label('Epoch')

        fig.suptitle(task_names[task_idx], y=1.02)
        sns.despine(fig=fig, offset=0, trim=False)
        fig.tight_layout()

        if save_figs:
            fig.savefig(os.path.join(outdir, f'trajectory_joint_{task_names[task_idx]}.svg'),
                        dpi=300, bbox_inches='tight', pad_inches=0.01)
        plt.show()

# Combined panel: accuracy, similarity, fMRI VE

In [ ]:
# Three-row panel: accuracy, weight-kernel similarity, fMRI VE — all vs epoch
# Columns = tasks, rows = metrics, one line per kernel

has_fmri = fmri_baselines['task'] is not None
n_metric_rows = 3 if has_fmri else 2
wk_metric = 'spearman'
fmri_key = 'fmri_task_ve'

size_scale = 2.0
font_size = int(6 * size_scale)
set_font_size(font_size)
fig_w = 3 * n_fig_columns * size_scale
fig_h = 2 * n_metric_rows * size_scale

fig, axes = plt.subplots(n_metric_rows, n_fig_columns,
                          figsize=(fig_w, fig_h),
                          squeeze=False, sharex=True)

row_labels = ['Test Accuracy', f'Weight–kernel similarity\n({wk_metric})']
if has_fmri:
    row_labels.append('Task fMRI VE\n(frac. of baseline)')

for model_idx in range(n_models):
    res = results[model_idx]
    if res is None:
        continue

    this = model_params.iloc[model_idx]
    task_index = this.task_index
    kernel_index = this.kernel_index
    col = task_index

    epochs = np.array(res['sampled_epochs'])
    n_runs = res['n_runs']

    # extract per-run arrays
    acc = np.zeros((n_runs, len(epochs)))
    wk = np.full((n_runs, len(epochs)), np.nan)
    ve = np.full((n_runs, len(epochs)), np.nan)

    has_wk = res['runs'][0]['epochs'][epochs[0]]['weight_kernel_similarity'] is not None

    for ri, run in enumerate(res['runs']):
        for ei, ep in enumerate(epochs):
            ep_data = run['epochs'][ep]
            acc[ri, ei] = ep_data['accuracy']
            if has_wk:
                wk[ri, ei] = ep_data['weight_kernel_similarity'][wk_metric]
            if has_fmri and ep_data[fmri_key] is not None:
                ve[ri, ei] = ep_data[fmri_key].sum(axis=0).mean() / fmri_baselines['task']

    # plot each metric row
    plot_data = [acc]
    if has_wk:
        plot_data.append(wk)
    else:
        plot_data.append(None)
    if has_fmri:
        plot_data.append(ve)

    for row, vals in enumerate(plot_data):
        if vals is None:
            continue
        ax = axes[row, col]
        m = np.nanmean(vals, axis=0)
        se = np.nanstd(vals, axis=0) / np.sqrt(n_runs)
        ax.plot(epochs, m, color=color_palette[kernel_index],
                label=this.kernel_label, linewidth=2)
        ax.fill_between(epochs, m - 1.96 * se, m + 1.96 * se,
                         color=color_palette[kernel_index], alpha=0.1)

# labels, titles, legend
for col in range(n_fig_columns):
    if col < n_tasks:
        axes[0, col].set_title(task_names[col])
    axes[-1, col].set_xlabel('Epoch')

for row in range(n_metric_rows):
    axes[row, 0].set_ylabel(row_labels[row])

axes[0, 0].set_ylim((-0.05, 1.05))

handles, labels = axes[0, 0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[0, 0].legend(by_label.values(), by_label.keys(), loc='lower right')

sns.despine(fig=fig, offset=0, trim=False, left=False, right=True, top=True, bottom=False)
fig.tight_layout()

if save_figs:
    fig.savefig(os.path.join(outdir, 'trajectory_combined_panel.svg'),
                dpi=300, bbox_inches='tight', pad_inches=0.01)
plt.show()